## Cal Poly, San Luis Obispo, Fall 2026
## DATA 3301: Introduction to Data Science

# Group Project: Data Notebook

## Before You Do Anything Else!!!

**At least one person** should save a copy of this notebook to their Google Drive (File > Save a copy in Drive) and share it with the group (Share, top right). Colab does not merge edits well, so avoid having two people run or edit the notebook at the same time.

**Project title:** ______________________________________

**Group members:** Robert Lively, Myanh Nguyen

**Our question (one sentence):** ______________________________________

**Data source(s), with links:** ______________________________________

**One row of our data represents:** ______________________________________

# How to use this notebook

This is a workspace, not a lab. It walks through the order of work from Labs 1 to 5: **load → prepare → describe one variable at a time → describe pairs → compute metrics → model**. Each part has a short prompt, a small reference table, and empty cells. Add as many cells as you need, and delete prompts that don't apply to your data.

* Make every chart with **Altair**, and make it **professional-looking**: labeled axes with units, a title, sensible scales, and readable legends.
* End each code cell by **displaying** what you computed, so you (and I) can review it.
* Put written answers in the **Notes** text cells, not in code comments.

### Where to find help

* **Data frames, joins, unions, grouping:** DS02, DS03, LN04, Lab 1
* **Categorical vs. numerical variables:** LN05, DS04
* **Altair:** DS06, DS07, and Lab 3's Altair How-To Guide (G0 to G11)
* **Quantitative variables, distance and similarity:** LN07, LN08, DS08
* **Text data (bag of words, TF-IDF):** LN09, LN10, DS09
* **One-hot encoding, KNN, linear regression:** LN11, LN12, DS11, DS12, Lab 4
* **Evaluation and model selection:** LN13, LN14, DS13, DS14, Lab 5

In [3]:
import pandas as pd
import numpy as np
import altair as alt

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold

### Add any other import statements you need below

# Part 1: Load Your Data

Four common ways to get data into a DataFrame. They are **examples only** (commented out). Copy the one you need into the cell at the end of Part 1, uncomment it, and fill in your own file names, URLs, and IDs.

**A. A file you have (CSV, Excel, ...)**

In [2]:
# 1. Click the folder icon in the left sidebar and drag your file in.
#    Files uploaded this way are DELETED when the session ends, so you'd re-upload every time.
# 2. Read it with pandas:
#
# df = pd.read_csv("my_data.csv")
# df = pd.read_excel("my_data.xlsx", sheet_name=0)    # 0 = first tab; or the tab's name
# df = pd.read_csv("my_data.tsv", sep="\t")           # tab-separated
#
# A file that is already online (a raw GitHub link, as in Lab 5) works the same way:
# df = pd.read_csv("https://raw.githubusercontent.com/<user>/<repo>/main/my_data.csv")
#
# To keep files between sessions, put them in Google Drive and mount it.
# (If a groupmate shared the folder with you, first right-click it in Drive > Organize > Add shortcut to My Drive.)
# from google.colab import drive
# drive.mount("/content/drive")
# df = pd.read_csv("/content/drive/MyDrive/<folder>/my_data.csv")

**B. A web API that returns JSON**

In [ ]:
# Read the API's documentation for: the URL (the "endpoint"), its query parameters, and whether it needs a key.
#
# url = "https://api.example.com/v1/records?city=San%20Luis%20Obispo&limit=1000"
# raw = pd.read_json(url, typ="series")     # the whole response
# raw.index                                 # what's inside? the records are usually under one key
# df = pd.json_normalize(raw["results"])    # one row per record; nested fields become "a.b" columns
#
# If the response is already a plain list of records, pd.read_json(url) gives you a DataFrame directly.
#
# API keys: never paste a key into the notebook (you will share it). Click the key icon in the
# left sidebar ("Secrets"), add it as MY_API_KEY, and allow notebook access. Then:
# from google.colab import userdata
# key = userdata.get("MY_API_KEY")
# df = pd.read_json(f"https://api.example.com/v1/records?api_key={key}")                      # key in the URL, or
# df = pd.read_json("https://api.example.com/v1/records", storage_options={"Authorization": f"Bearer {key}"})   # key in a header
#
# Many APIs return one "page" (say, 100 records) per request; check the docs for limit/offset or page parameters.
# Save what you pulled so you don't hit the API on every run:
# df.to_csv("my_data.csv", index=False)

**C. A Google Sheet your group edits together**

In [ ]:
# In the sheet: Share > General access > "Anyone with the link" (Viewer). Then copy the IDs from its URL:
#   https://docs.google.com/spreadsheets/d/<SHEET_ID>/edit#gid=<TAB_ID>
#
# sheet_id = "<SHEET_ID>"
# tab_id = 0                                # the gid of the tab you want; the first tab is usually 0
# df = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={tab_id}")

**D. A table on a web page (Wikipedia, a government site, ...)**

In [ ]:
# read_html returns a LIST with every <table> on the page. Find the one you want.
#
# tables = pd.read_html("https://en.wikipedia.org/wiki/<Page_Title>")
# len(tables)
# df = tables[0]
#
# Narrow it down with text that appears in your table:      pd.read_html(url, match="Population")
# If the site refuses the request (HTTP 403 Forbidden), add:  storage_options={"User-Agent": "Mozilla/5.0"}

**Your data:** load every table your project needs.

In [ ]:
# Load your data here

# Part 2: Prepare Your Data

Get to **one DataFrame where each row is one observation** (your answer to "one row represents...") and each column is one variable you will use.

| You want to... | Code |
|---|---|
| look at it | `df.shape`, `df.dtypes`, `df.head()`, `df.isna().sum()` |
| fix a type | `pd.to_numeric(df["col"], errors="coerce")`, `pd.to_datetime(df["col"])`, `df["col"].astype("category")` |
| keep the columns you need | `df[["col1", "col2"]]`, `df.rename(columns={...})` |
| keep the rows you need | `df.query("year >= 2020")`, `df.dropna(subset=[...])` |
| combine tables | `df1.merge(df2, on="key", how="inner")` (join), `pd.concat([df1, df2])` (union) |
| change what a row is | `df.groupby("col").agg(...)`, `df.melt(...)`, `df.pivot_table(...)` |

In [ ]:
# 2.1 Inspect

In [ ]:
# 2.2 Clean (types, missing values, odd values)

In [ ]:
# 2.3 Project: keep the columns (and rows) you need

In [ ]:
# 2.4 Combine tables / change what a row is (if needed)

**Notes:** What did you drop, fix, or combine, and why? Which columns are categorical and which are numerical?

# Part 3: One Variable at a Time

For each variable you care about: **numerical** → summary statistics and a histogram; **categorical** → counts and a bar chart. Look for skew, outliers, odd values, and rare categories.

| You want to... | Code |
|---|---|
| summarize | `df.describe()`, `df["col"].value_counts()` |
| one histogram | `alt.Chart(df).mark_bar().encode(alt.X("col:Q", bin=alt.Bin(maxbins=30), title="..."), alt.Y("count()", title="Count"))` |
| a histogram for each of several columns | `alt.Chart(df).mark_bar().encode(alt.X(alt.repeat("repeat"), type="quantitative", bin=True), y="count()").repeat(repeat=num_cols, columns=3)` |
| a bar chart of counts | `alt.Chart(df).mark_bar().encode(alt.X("count()"), alt.Y("col:N", sort="-x"))` |

Altair refuses data with more than 5,000 rows. Chart a sample (`df.sample(5000, random_state=0)`) or run `alt.data_transformers.disable_max_rows()` (your exported HTML gets large).

In [ ]:
# 3.1 Summary statistics

In [ ]:
# 3.2 Histograms of numerical variables

In [ ]:
# 3.3 Counts of categorical variables

**Notes:** What do the distributions look like? Anything surprising or that needs cleaning back in Part 2?

# Part 4: Two Variables at a Time

Look at how pairs of variables relate, especially each candidate feature against the variable you want to predict.

| You want to... | Code |
|---|---|
| a scatter plot | `alt.Chart(df).mark_circle().encode(alt.X("a:Q", title="..."), alt.Y("b:Q", title="..."), alt.Color("group:N", title="..."))` |
| a scatter plot matrix | `alt.Chart(df).mark_circle().encode(alt.X(alt.repeat("column"), type="quantitative"), alt.Y(alt.repeat("row"), type="quantitative")).repeat(row=cols, column=cols)` |
| numerical vs. categorical | `alt.Chart(df).mark_boxplot().encode(alt.X("group:N"), alt.Y("value:Q"))` |
| correlations | `df.corr(numeric_only=True)`, `df[cols].corrwith(df["target"]).sort_values()` |

In [ ]:
# 4.1 Scatter plots

In [ ]:
# 4.2 Correlations

**Notes:** Which relationships look strong, weak, or non-linear? Which variables look useful for predicting your target?

# Part 5: Compute Your Metrics

Compute the numbers your question depends on. Depending on your project, that may be group summaries, rates or ratios you derive, standardized scores, or distances and similarities between observations.

| You want to... | Code |
|---|---|
| summarize by group | `df.groupby("group")["col"].agg(["mean", "median", "std", "count"])` |
| a new derived column | `df.assign(rate=df["count"] / df["total"])` |
| standardize (z-scores) | `(df[cols] - df[cols].mean()) / df[cols].std()` |
| distances / similarities between rows | `from sklearn.metrics.pairwise import pairwise_distances, cosine_similarity` |
| text into numbers | `from sklearn.feature_extraction.text import TfidfVectorizer` |

In [ ]:
# 5.1

In [ ]:
# 5.2

**Notes:** What do these numbers tell you about your question?

# Part 6: Build and Test Models

Pick a **target** (`y`) and **features** (`X`). Start with a baseline, try more than one method, grade every model with **cross-validation on the same folds**, tune hyperparameters, and compare.

If your target is **categorical**, set up `X` and `y` now; the classification versions (`DummyClassifier`, `KNeighborsClassifier`, `LogisticRegression`, scored with accuracy) come in weeks 10 and 11.

| You want to... | Code |
|---|---|
| hold out a test set (optional "lockbox") | `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)` |
| cross-validated RMSE | `np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=folds).mean())` |
| training RMSE | `np.sqrt(mean_squared_error(y, model.fit(X, y).predict(X)))` |
| one-hot encode + scale inside a model | `make_pipeline(make_column_transformer((OneHotEncoder(sparse_output=False), cat_cols), (StandardScaler(), num_cols)), KNeighborsRegressor())` |
| grid search | `GridSearchCV(model, param_grid={"kneighborsregressor__n_neighbors": range(1, 31)}, scoring="neg_mean_squared_error", cv=folds).fit(X, y)` |
| read a grid search | `.best_params_`, `np.sqrt(-.best_score_)`, `pd.DataFrame(.cv_results_)` |

In [ ]:
# Use these folds for every cross-validation, so your scores are comparable.
folds = KFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
# 6.1 Target (y) and features (X)

In [ ]:
# 6.2 Baseline

In [ ]:
# 6.3 Model 1

In [ ]:
# 6.4 Model 2

In [ ]:
# 6.5 Tune hyperparameters

In [ ]:
# 6.6 Compare: one table and one Altair chart of CV RMSE for every model (baseline included)

**Notes:** Which model would you recommend, and how much better than the baseline is it? What would you try next?

# Part 7: Where You Are

**Notes:** What have you learned so far? What still needs work, and who is doing it? What do you need help with?

## Submission Instructions

To share this notebook (for example, as proof of your preliminary work), export it as an **HTML file**. No PDF conversion needed.

- Restart this notebook and run the cells from beginning to end.
  - Go to Runtime > Restart session and run all.
  - Make sure every cell ran without errors and every chart appears.
- Run the cell below. It downloads this notebook as a webpage, `_NOTEBOOK.html`.

In [ ]:
# @markdown Run this cell to download this notebook as a webpage, `_NOTEBOOK.html`.

import google, json, nbformat

# Get the current notebook and write it to _NOTEBOOK.ipynb
raw_notebook = google.colab._message.blocking_request("get_ipynb",
                                                      timeout_sec=30)["ipynb"]
with open("_NOTEBOOK.ipynb", "w", encoding="utf-8") as ipynb_file:
  ipynb_file.write(json.dumps(raw_notebook))

# Use nbconvert to convert .ipynb to .html.
!jupyter nbconvert --to html --log-level WARN _NOTEBOOK.ipynb

# Download the .html file.
google.colab.files.download("_NOTEBOOK.html")

- Open `_NOTEBOOK.html` in your browser and check that all of your code, output, written answers, and charts are there. (The charts load from the internet, so you need to be online for them to appear.)